In [1]:
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, acf, pacf
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from itertools import product
import warnings
warnings.filterwarnings("ignore")

In [2]:
CSV_FILE = "training_data.csv"
PERIOD   = 4
N_LAGS   = 4
MODELS   = ["SARIMA", "ETS", "BSTS", "RandomForest", "GradientBoosting"]

In [3]:
# rmse
def compute_rmse(actual, predicted):
    actual    = np.array(actual, dtype=float)
    predicted = np.array(predicted, dtype=float)
    if len(actual) == 0: return np.nan
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))

In [4]:
# mape
def compute_mape(actual, predicted):
    actual    = np.array(actual, dtype=float)
    predicted = np.array(predicted, dtype=float)
    mask = actual != 0
    if mask.sum() == 0: return np.nan
    return float(np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100)


In [5]:
# sarima
def run_sarima(train_arr, val_arr, period=4):
    try:
        train = pd.Series(train_arr.astype(float))
        try:
            d = 0 if adfuller(train.dropna(), autolag="AIC")[1] < 0.05 else 1
        except:
            d = 1
        try:
            diff_s = train.diff(period).dropna()
            D = 0 if (len(diff_s) > 5 and adfuller(diff_s, autolag="AIC")[1] < 0.05) else 1
        except:
            D = 1
        try:
            T  = len(train)
            ci = 1.96 / np.sqrt(T)
            lm = min(20, T // 2 - 1)
            acf_v  = acf(train.dropna(),  nlags=lm, fft=True)
            pacf_v = pacf(train.dropna(), nlags=lm, method="ywmle")
            pmax = min(max((k for k in range(1, len(pacf_v)) if abs(pacf_v[k]) > ci), default=1), 3)
            qmax = min(max((k for k in range(1, len(acf_v))  if abs(acf_v[k])  > ci), default=1), 3)
            Pmax = 1 if any(abs(pacf_v[k]) > ci for k in range(period, len(pacf_v), period)) else 0
            Qmax = 1 if any(abs(acf_v[k])  > ci for k in range(period, len(acf_v),  period)) else 0
        except:
            pmax, qmax, Pmax, Qmax = 1, 1, 1, 1

        best_aic, best_fit = np.inf, None
        for p, q, P, Q in product(range(pmax+1), range(qmax+1),
                                   range(Pmax+1), range(Qmax+1)):
            if p == q == P == Q == 0: continue
            try:
                fit = SARIMAX(train, order=(p,d,q),
                              seasonal_order=(P,D,Q,period),
                              enforce_stationarity=False,
                              enforce_invertibility=False).fit(disp=False)
                all_sig = all(abs(fit.params / fit.bse) > 1.96)
                if all_sig and fit.aic < best_aic:
                    best_aic, best_fit = fit.aic, fit
            except: pass

        if best_fit is None:
            for p, q, P, Q in product(range(pmax+1), range(qmax+1),
                                       range(Pmax+1), range(Qmax+1)):
                if p == q == P == Q == 0: continue
                try:
                    fit = SARIMAX(train, order=(p,d,q),
                                  seasonal_order=(P,D,Q,period),
                                  enforce_stationarity=False,
                                  enforce_invertibility=False).fit(disp=False)
                    if fit.aic < best_aic:
                        best_aic, best_fit = fit.aic, fit
                except: pass

        if best_fit is None: return np.nan, np.nan
        pred = best_fit.get_forecast(steps=len(val_arr)).predicted_mean.clip(lower=0).values
        return compute_rmse(val_arr, pred), compute_mape(val_arr, pred)
    except:
        return np.nan, np.nan



In [6]:
# ets
def run_ets(train_arr, val_arr, m=4):
    try:
        y = train_arr.astype(float)
        if len(y) < m * 2: return np.nan, np.nan

        def _ets_fit(y, alpha, beta, gamma, m):
            n = len(y)
            l = np.zeros(n); b = np.zeros(n); s = np.zeros(n + m)
            l[0] = np.mean(y[:m])
            b[0] = (np.mean(y[min(m,n-1):min(2*m,n)]) - np.mean(y[:m])) / m
            for i in range(m): s[i] = y[i] - np.mean(y[:m])
            y_hat = np.zeros(n); y_hat[0] = l[0] + b[0] + s[0]
            for t in range(1, n):
                l_prev=l[t-1]; b_prev=b[t-1]; s_prev=s[t]
                l[t]   = alpha*(y[t]-s_prev) + (1-alpha)*(l_prev+b_prev)
                b[t]   = beta*(l[t]-l[t-1])  + (1-beta)*b_prev
                s[t+m] = gamma*(y[t]-l[t])   + (1-gamma)*s_prev
                y_hat[t] = l[t]+b[t]+(s[t+m-m+1] if t+1<n else 0)
            return y_hat, l, b, s[m:m+n]

        best_sse, best_params = np.inf, (0.3, 0.1, 0.2)
        for a in np.arange(0.05, 0.95, 0.1):
            for b in np.arange(0.05, 0.95, 0.1):
                for g in np.arange(0.05, 0.95, 0.1):
                    try:
                        yh, _, _, _ = _ets_fit(y, a, b, g, m)
                        sse = np.sum((y - yh)**2)
                        if sse < best_sse: best_sse=sse; best_params=(a,b,g)
                    except: pass

        a, b, g = best_params
        _, level, trend_c, seasonal = _ets_fit(y, a, b, g, m)

        preds = []
        for step in range(1, len(val_arr)+1):
            s_idx = -(m-(step%m)) if (step%m)!=0 else -m
            preds.append(level[-1] + step*trend_c[-1] + seasonal[s_idx])
        pred = np.array(preds)
        return compute_rmse(val_arr, pred), compute_mape(val_arr, pred)
    except:
        return np.nan, np.nan



In [7]:
def run_bsts(train_arr, val_arr, period=4):
    try:
        train = train_arr.astype(float); test = val_arr.astype(float)
        n = len(train); h = len(test)
        if h == 0 or n < period + 2: return np.nan, np.nan

        var_y  = max(np.var(np.diff(train), ddof=1) if n > 1 else 1.0, 1e-6)
        slope0 = float(np.mean(np.diff(train[:min(4,n)]))) if n > 1 else 0.0

        def kalman(y, F, H, Q, R, x0, P0):
            d=len(x0); x=x0.copy(); P=P0.copy(); states=np.zeros((len(y),d))
            for t in range(len(y)):
                xp=F@x; Pp=F@P@F.T+Q; S=float(H@Pp@H.T)+R
                K=(Pp@H.T)/S; x=xp+K.flatten()*(y[t]-float(H@xp))
                P=(np.eye(d)-np.outer(K.flatten(),H.flatten()))@Pp; states[t]=x
            return states

        def kforecast(states, F, H, h):
            x=states[-1].copy(); preds=[]
            for _ in range(h): x=F@x; preds.append(float(H@x))
            return np.array(preds)

        def inrmse(states, H, y):
            return float(np.sqrt(np.mean((y-np.array([float(H@s) for s in states]))**2)))

        F1=np.array([[1.]]); H1=np.array([[1.]]); Q1=np.array([[var_y*0.01]])
        R1=var_y*0.1; x01=np.array([train[0]]); P01=np.array([[var_y]])

        F2=np.array([[1.,1.],[0.,1.]]); H2=np.array([[1.,0.]])
        Q2=np.diag([var_y*0.01,var_y*0.001]); R2=var_y*0.1
        x02=np.array([train[0],slope0]); P02=np.diag([var_y,var_y*0.1])

        d3=2+(period-1); F3=np.zeros((d3,d3))
        F3[0,0]=1.; F3[0,1]=1.; F3[1,1]=1.; F3[2,2:]=-1.
        for i in range(3,d3): F3[i,i-1]=1.
        H3=np.zeros((1,d3)); H3[0,0]=1.; H3[0,2]=1.
        Q3=np.zeros((d3,d3)); Q3[0,0]=var_y*0.01; Q3[1,1]=var_y*0.001; Q3[2,2]=var_y*0.01
        R3=var_y*0.1
        si=np.zeros(period-1)
        if n>=period:
            for s in range(period-1):
                v=[train[t]-np.mean(train[:period]) for t in range(s,n,period) if t<n]
                if v: si[s]=float(np.mean(v))
        x03=np.concatenate([[train[0]],[slope0],si])
        P03=np.diag([var_y]+[var_y*0.1]+[var_y*0.5]*(period-1))

        F4=np.array([[1.,1.],[0.,0.8]]); H4=np.array([[1.,0.]])
        Q4=np.diag([var_y*0.01,var_y*0.001]); R4=var_y*0.1
        x04=np.array([train[0],slope0]); P04=np.diag([var_y,var_y*0.1])

        specs=[(F1,H1,Q1,R1,x01,P01),(F2,H2,Q2,R2,x02,P02),
               (F3,H3,Q3,R3,x03,P03),(F4,H4,Q4,R4,x04,P04)]

        best_r=np.inf; best_st=best_F=best_H=None
        for F,H,Q,R,x0,P0 in specs:
            try:
                st=kalman(train,F,H,Q,R,x0,P0); r=inrmse(st,H,train)
                if r<best_r: best_r=r; best_st=st; best_F=F; best_H=H
            except: continue

        if best_st is None: return np.nan, np.nan
        pred = kforecast(best_st, best_F, best_H, h)
        return compute_rmse(test, pred), compute_mape(test, pred)
    except:
        return np.nan, np.nan


In [8]:
# random forest
def run_random_forest(train_arr, val_arr, n_lags=4):
    try:
        train=train_arr.astype(float); test=val_arr.astype(float)
        if len(train)<=n_lags or len(test)==0: return np.nan, np.nan
        X_tr=np.array([train[i-n_lags:i][::-1] for i in range(n_lags,len(train))])
        y_tr=train[n_lags:]
        m=RandomForestRegressor(n_estimators=200,max_features="sqrt",
                                min_samples_leaf=5,oob_score=True,random_state=42,n_jobs=-1)
        m.fit(X_tr,y_tr)
        history,preds=list(train),[]
        for _ in range(len(test)):
            p=m.predict([np.array(history[-n_lags:])[::-1]])[0]
            preds.append(p); history.append(p)
        pred=np.array(preds)
        return compute_rmse(test,pred), compute_mape(test,pred)
    except:
        return np.nan, np.nan

In [9]:
# gradient boosting
def run_gradient_boosting(train_arr, val_arr, n_lags=4):
    try:
        train=train_arr.astype(float); test=val_arr.astype(float)
        if len(train)<=n_lags or len(test)==0: return np.nan, np.nan
        X_tr=np.array([train[i-n_lags:i][::-1] for i in range(n_lags,len(train))])
        y_tr=train[n_lags:]
        m=GradientBoostingRegressor(n_estimators=500,learning_rate=0.05,
                                    max_depth=3,min_samples_leaf=5,
                                    subsample=0.8,loss="squared_error",random_state=42)
        m.fit(X_tr,y_tr)
        history,preds=list(train),[]
        for _ in range(len(test)):
            p=m.predict([np.array(history[-n_lags:])[::-1]])[0]
            preds.append(p); history.append(p)
        pred=np.array(preds)
        return compute_rmse(test,pred), compute_mape(test,pred)
    except:
        return np.nan, np.nan

In [ ]:
# run all models
df_raw   = pd.read_csv(CSV_FILE)
series_cols = [c for c in df_raw.columns if c != "ds"]

print(f"Loaded {len(series_cols)} series\n")
print("=" * 70)

records = []

for sid in series_cols:
    series = df_raw[sid].dropna().values
    n      = len(series)
    split  = int(n * 0.8)

    if split < N_LAGS + 4 or split >= n:
        print(f"  Skipping {sid} — too short ({n} obs)")
        continue

    train_arr = series[:split]
    val_arr   = series[split:]

    rmse_s, mape_s = run_sarima(train_arr, val_arr, PERIOD)
    rmse_e, mape_e = run_ets(train_arr, val_arr, PERIOD)
    rmse_b, mape_b = run_bsts(train_arr, val_arr, PERIOD)
    rmse_r, mape_r = run_random_forest(train_arr, val_arr, N_LAGS)
    rmse_g, mape_g = run_gradient_boosting(train_arr, val_arr, N_LAGS)

    rmse_dict = {"SARIMA": rmse_s, "ETS": rmse_e, "BSTS": rmse_b,
                 "RandomForest": rmse_r, "GradientBoosting": rmse_g}
    mape_dict = {"SARIMA": mape_s, "ETS": mape_e, "BSTS": mape_b,
                 "RandomForest": mape_r, "GradientBoosting": mape_g}

    valid_rmse = {k: v for k, v in rmse_dict.items() if not np.isnan(v)}
    valid_mape = {k: v for k, v in mape_dict.items() if not np.isnan(v)}

    best_rmse = min(valid_rmse, key=valid_rmse.get) if valid_rmse else np.nan
    best_mape = min(valid_mape, key=valid_mape.get) if valid_mape else np.nan

    record = {
        "sample":               sid,
        "rmse_SARIMA":          round(rmse_s, 4) if not np.isnan(rmse_s) else np.nan,
        "rmse_ETS":             round(rmse_e, 4) if not np.isnan(rmse_e) else np.nan,
        "rmse_BSTS":            round(rmse_b, 4) if not np.isnan(rmse_b) else np.nan,
        "rmse_RandomForest":    round(rmse_r, 4) if not np.isnan(rmse_r) else np.nan,
        "rmse_GradientBoosting":round(rmse_g, 4) if not np.isnan(rmse_g) else np.nan,
        "mape_SARIMA":          round(mape_s, 4) if not np.isnan(mape_s) else np.nan,
        "mape_ETS":             round(mape_e, 4) if not np.isnan(mape_e) else np.nan,
        "mape_BSTS":            round(mape_b, 4) if not np.isnan(mape_b) else np.nan,
        "mape_RandomForest":    round(mape_r, 4) if not np.isnan(mape_r) else np.nan,
        "mape_GradientBoosting":round(mape_g, 4) if not np.isnan(mape_g) else np.nan,
        "min_rmse":      best_rmse,
        "min_mape":      best_mape,
    }
    records.append(record)
    print(f"  {sid:<12}  best_rmse={best_rmse:<20}  best_mape={best_mape}")

# ============================================================
# SAVE
# ============================================================

results_df = pd.DataFrame(records)
results_df.to_csv("O_selected_models.csv", index=False)

print("\n  Best model distribution (RMSE):")
print(results_df["min_rmse"].value_counts().to_string())
print("\n  Best model distribution (MAPE):")
print(results_df["min_mape"].value_counts().to_string())

Loaded 3000 series

  series_0001   best_rmse=ETS                   best_mape=ETS
  series_0002   best_rmse=ETS                   best_mape=BSTS
  series_0003   best_rmse=ETS                   best_mape=ETS
  series_0004   best_rmse=RandomForest          best_mape=RandomForest
  series_0005   best_rmse=ETS                   best_mape=ETS
  series_0006   best_rmse=GradientBoosting      best_mape=GradientBoosting
  series_0007   best_rmse=SARIMA                best_mape=RandomForest
  series_0008   best_rmse=RandomForest          best_mape=RandomForest
  series_0009   best_rmse=RandomForest          best_mape=RandomForest
  series_0010   best_rmse=BSTS                  best_mape=BSTS
  series_0011   best_rmse=GradientBoosting      best_mape=GradientBoosting
  series_0012   best_rmse=ETS                   best_mape=ETS
  series_0013   best_rmse=SARIMA                best_mape=BSTS
  series_0014   best_rmse=BSTS                  best_mape=BSTS
  series_0015   best_rmse=BSTS                

keep only series with same ranking in mape and rmse values, like if the order of mape is bsts mape < ets mape < sarima mape < gradient boosting mape < random forest mape, then rmse should be in the same order. If not, then remove that series from the df1, that allows to filter series with clear patterns and features, so the mesure of the fixed score matrix in the BWM would be more accurate.

In [ ]:
# keep only models with same ranking in mape rmse values like if the order of mape is bsts mape <ets mape < sarima mape< gradient boosting mape < random forest map then rmse should be in the same order. If not, then remove that series from the df1

df1 = pd.read_csv("O_selected_models.csv")

models = ["SARIMA", "ETS", "BSTS", "RandomForest", "GradientBoosting"]

def same_ranking(row):
    mape_order = sorted(models, key=lambda m: row[f"mape_{m}"])
    rmse_order = sorted(models, key=lambda m: row[f"rmse_{m}"])

    return mape_order == rmse_order

df1 = df1[df1.apply(same_ranking, axis=1)]

df1.to_csv("O_selected_models.csv", index=False)

In [ ]:
dff = pd.read_csv("O_selected_models.csv")
print("\n  Best model distribution (RMSE):")
print(dff["min_rmse"].value_counts().to_string())
print("\n  Best model distribution (MAPE):")
print(dff["min_mape"].value_counts().to_string())


  Best model distribution (RMSE):
min_rmse
BSTS                570
ETS                 460
SARIMA              432
RandomForest        344
GradientBoosting    232

  Best model distribution (MAPE):
min_mape
BSTS                570
ETS                 460
SARIMA              432
RandomForest        344
GradientBoosting    232
